# Quant Crypto Price Aggregation Demo

This notebook uses synthetic data only (no confidential real dataset).

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root / 'src'))

from quant_aggregator.methods import add_all_aggregators
from quant_aggregator.metrics import rmse_by_aggregator
from quant_aggregator.plotting import plot_prices_by_exchange
from quant_aggregator.simulation import simulate_fragmented_market


In [ ]:
trades = simulate_fragmented_market(
    n_steps=1440,
    exchanges=('emolas', 'niamor', 'onurb', 'omit', 'xela'),
    initial_price=100.0,
    drift=4.0,
    volatility=2.0,
    copula_theta=4.0,
    outlier_prob=0.3,
    normal_sigma=0.025,
    outlier_df=2,
    outlier_scale=0.15,
    ar_coeff=0.8,
    heavy_tail_volumes=True,
    volume_scale=30.0,
    burr_params=(1.0, 4.0),
    seed=42,
)

trades['timestamp'] = trades['timestamp'].astype('datetime64[ns]')
trades = trades.sort_values('timestamp').set_index('timestamp')
add_all_aggregators(trades)

rmse_scores = rmse_by_aggregator(trades, target_col='efficient_price')
rmse_scores


In [ ]:
plot_prices_by_exchange(trades, add_sgrd=True, add_efficient=True)

SGRD note: useful on stressed days such as `btc-eur-2023-03-05`,
where cross-exchange price dispersion can appear between large and smaller venues.